In [1]:
from pymongo import MongoClient
import pandas as pd
from pprint import pprint

client = MongoClient("mongodb://127.0.0.1:27017/")
mongo_db = client["LetsMeet"]
users = mongo_db["users"]

print("Dokumente:", users.count_documents({}))

<jemalloc>: Unsupported system page size


Dokumente: 1576


In [2]:
document = users.find_one()

print("Felder und Datentypen:")
for key, value in document.items():
    print(f"{key:12} {type(value).__name__:12} {repr(value)[:150]}")

Felder und Datentypen:
_id          str          'Martin.Forster@web.ork'
name         str          'Forster, Martin'
phone        str          '02372 8020'
friends      list         []
likes        list         []
messages     list         []
createdAt    str          '2023-12-21T14:45:02'
updatedAt    str          '2024-09-02T17:13:38'


In [3]:
all_fields = set()

for document in users.find():
    all_fields.update(document.keys())

print("Alle Felder:")
for field in sorted(all_fields):
    print(field)

Alle Felder:
_id
createdAt
friends
likes
messages
name
phone
updatedAt


In [4]:
like_example = users.find_one(
    {"likes.0": {"$exists": True}},
    {"_id": 1, "likes": 1}
)

message_example = users.find_one(
    {"messages.0": {"$exists": True}},
    {"_id": 1, "messages": 1}
)

print("Beispiel Likes:")
pprint(like_example)

print("\nBeispiel Nachrichten:")
pprint(message_example)

Beispiel Likes:
{'_id': 'ellen.bäumker@web.ork',
 'likes': [{'liked_email': 'gerlind.feuerhahn@1mal1.kom',
            'status': 'mutual',
            'timestamp': '2024-04-17 19:49:47'}]}

Beispiel Nachrichten:
{'_id': 'karl-heinz.fink@ge-em-ix.kom',
 'messages': [{'conversation_id': 47,
               'message': 'Wollen wir das Projekt gemeinsam angehen?',
               'receiver_email': 'lispector.carola@gmaiil.kom',
               'timestamp': '2024-08-10 12:01:07'}]}


In [5]:
print(
    "Dokumente mit Freunden:",
    users.count_documents({"friends.0": {"$exists": True}})
)

print(
    "Dokumente mit Likes:",
    users.count_documents({"likes.0": {"$exists": True}})
)

print(
    "Dokumente mit Nachrichten:",
    users.count_documents({"messages.0": {"$exists": True}})
)

Dokumente mit Freunden: 0
Dokumente mit Likes: 430
Dokumente mit Nachrichten: 270


In [6]:
mongo_documents = list(users.find())

total_likes = sum(len(document.get("likes", [])) for document in mongo_documents)
total_messages = sum(len(document.get("messages", [])) for document in mongo_documents)
total_friends = sum(len(document.get("friends", [])) for document in mongo_documents)

print("Likes insgesamt:", total_likes)
print("Nachrichten insgesamt:", total_messages)
print("Freundschaftseinträge insgesamt:", total_friends)

Likes insgesamt: 500
Nachrichten insgesamt: 300
Freundschaftseinträge insgesamt: 0


In [7]:
mongo_emails_lower = {
    document["_id"].lower()
    for document in mongo_documents
}

unknown_like_references = []
unknown_message_references = []

for document in mongo_documents:
    sender_email = document["_id"]

    for like in document.get("likes", []):
        liked_email = like["liked_email"]

        if liked_email.lower() not in mongo_emails_lower:
            unknown_like_references.append(
                (sender_email, liked_email)
            )

    for message in document.get("messages", []):
        receiver_email = message["receiver_email"]

        if receiver_email.lower() not in mongo_emails_lower:
            unknown_message_references.append(
                (sender_email, receiver_email)
            )

print("Unbekannte Like-Ziele:", len(unknown_like_references))
print("Unbekannte Nachrichten-Empfänger:", len(unknown_message_references))

Unbekannte Like-Ziele: 0
Unbekannte Nachrichten-Empfänger: 0


In [8]:
from pathlib import Path

excel_path = Path("../Lets Meet DB Dump.xlsx")

excel_df = pd.read_excel(
    excel_path,
    sheet_name="Tabelle1",
    dtype=str,
    keep_default_na=False,
)

mongo_profiles = pd.DataFrame([
    {
        "mongo_email": document["_id"],
        "mongo_name": document.get("name", ""),
        "mongo_phone": document.get("phone", ""),
    }
    for document in mongo_documents
])

excel_profiles = pd.DataFrame({
    "excel_email": excel_df["E-Mail"],
    "excel_name": excel_df["Nachname, Vorname"],
    "excel_phone": excel_df["Telefon"],
})

excel_profiles["email_lower"] = (
    excel_profiles["excel_email"].str.lower()
)

mongo_profiles["email_lower"] = (
    mongo_profiles["mongo_email"].str.lower()
)

comparison = excel_profiles.merge(
    mongo_profiles,
    on="email_lower",
    how="outer",
    indicator=True,
)

print(comparison["_merge"].value_counts())

case_differences = (
    comparison["excel_email"] != comparison["mongo_email"]
)

name_conflicts = (
    comparison["excel_name"] != comparison["mongo_name"]
)

phone_conflicts = (
    comparison["excel_phone"] != comparison["mongo_phone"]
)

print("Abweichende E-Mail-Schreibweise:", case_differences.sum())
print("Namenskonflikte:", name_conflicts.sum())
print("Telefonkonflikte:", phone_conflicts.sum())

_merge
both          1576
left_only        0
right_only       0
Name: count, dtype: int64
Abweichende E-Mail-Schreibweise: 2
Namenskonflikte: 7
Telefonkonflikte: 3


In [9]:
display(
    comparison.loc[
        name_conflicts | phone_conflicts,
        [
            "excel_email",
            "excel_name",
            "mongo_name",
            "excel_phone",
            "mongo_phone",
        ]
    ].head(20)
)

,excel_email,excel_name,mongo_name,excel_phone,mongo_phone
144,benoit.françoise@autluuk.kom,"Françoise, Benoit","Françoise, Benoit",0541 / 385576,0541 / 385576
181,birgit.voss@gmaiil.ork,"Voss, Birgit","Voss, Birgit",06171 / 62808,0
406,ernst.zumdohme@gmaiil.kom,"Zumdohme, Ernst","Zumdohme, Ernst",824696843,824696843
633,hosseini.schukria@ge-em-ix.te,"Schukria, Hosseini","Schukria, Hosseini",0711 / 755641,0711 / 755641
725,joyeux.jérôme@d-ohnline.te,"Jérôme, Joyeux","Jérôme, Joyeux",(0941) 264067,(0941) 264067
729,julia.nagel@d-ohnline.ork,"Nagel, Julia","Nagel, Julia",0531 / 638986,0531 / 771204
785,katharina.prommer@autluuk.kom,"Prommer, Katharina","Vogelsang, Katharina",0411 / 584472,0411 / 584472
1068,nazemi.nahid@d-ohnline.ork,"Nahid, Nazemi","Nahid, Nazemi",(069) 5724111,(069) 5724111
1177,qarizada.fariba@autluuk.ork,"Fariba, Qarizada","Fariba, Qarizada",089 1933647,089 1933647
1370,thibaudet.mikaël@web.kom,"Mikaël, Thibaudet","Mikaël, Thibaudet",0441 / 383695,0441 / 383695


In [10]:
conflicts = comparison.loc[
    name_conflicts | phone_conflicts,
    [
        "excel_email",
        "excel_name",
        "mongo_name",
        "excel_phone",
        "mongo_phone",
    ]
]

for _, row in conflicts.iterrows():
    print("E-Mail:", repr(row["excel_email"]))
    print("Excel-Name:", repr(row["excel_name"]))
    print("Mongo-Name:", repr(row["mongo_name"]))
    print("Excel-Telefon:", repr(row["excel_phone"]))
    print("Mongo-Telefon:", repr(row["mongo_phone"]))
    print("-" * 60)

E-Mail: 'benoit.françoise@autluuk.kom'
Excel-Name: 'Françoise, Benoit '
Mongo-Name: 'Françoise, Benoit'
Excel-Telefon: '0541 / 385576'
Mongo-Telefon: '0541 / 385576'
------------------------------------------------------------
E-Mail: 'birgit.voss@gmaiil.ork'
Excel-Name: 'Voss, Birgit'
Mongo-Name: 'Voss, Birgit'
Excel-Telefon: '06171 / 62808'
Mongo-Telefon: '0'
------------------------------------------------------------
E-Mail: 'ernst.zumdohme@gmaiil.kom'
Excel-Name: 'Zumdohme, Ernst'
Mongo-Name: 'Zumdohme, Ernst'
Excel-Telefon: ' 824696843'
Mongo-Telefon: '824696843'
------------------------------------------------------------
E-Mail: 'hosseini.schukria@ge-em-ix.te'
Excel-Name: 'Schukria, Hosseini '
Mongo-Name: 'Schukria, Hosseini'
Excel-Telefon: '0711 / 755641'
Mongo-Telefon: '0711 / 755641'
------------------------------------------------------------
E-Mail: 'joyeux.jérôme@d-ohnline.te'
Excel-Name: 'Jérôme, Joyeux '
Mongo-Name: 'Jérôme, Joyeux'
Excel-Telefon: '(0941) 264067'
Mongo-

In [11]:
display(
    comparison.loc[
        case_differences,
        ["excel_email", "mongo_email"]
    ]
)

,excel_email,mongo_email
935,maria.westendarp@iunitimail.te,Maria.Westendarp@iunitimail.te
975,martin.forster@web.ork,Martin.Forster@web.ork


In [12]:
interest_column = "Interessiert an"

print("Vorkommende Werte:")
print(excel_df[interest_column].value_counts())

print(
    "Unterschiedliche Werte:",
    sorted(excel_df[interest_column].unique())
)

Vorkommende Werte:
Interessiert an
w     908
m     635
mw     33
Name: count, dtype: int64
Unterschiedliche Werte: ['m', 'mw', 'w']


In [13]:
interest_rows = []

for _, row in excel_df.iterrows():
    email = row["E-Mail"]
    raw_codes = row["Interessiert an"]

    for interest_code in raw_codes:
        interest_rows.append({
            "email": email,
            "interest_code": interest_code,
        })

user_interests = pd.DataFrame(interest_rows)

print("Zeilen:", len(user_interests))
print(user_interests["interest_code"].value_counts())
print(
    "Doppelte Zuordnungen:",
    user_interests.duplicated(
        subset=["email", "interest_code"]
    ).sum()
)

Zeilen: 1609
interest_code
w    941
m    668
Name: count, dtype: int64
Doppelte Zuordnungen: 0


In [14]:
import re

hobby_column = excel_df.columns[3]
hobby_rows = []
invalid_hobbies = []

for row_number, row in excel_df.iterrows():
    email = row["E-Mail"]
    raw_hobbies = row[hobby_column]

    if raw_hobbies == "":
        continue

    for part in raw_hobbies.split("; "):
        if part == "":
            continue

        match = re.fullmatch(r"(.+) %(-?\d+)%", part)

        if match is None:
            invalid_hobbies.append({
                "row": row_number,
                "email": email,
                "value": part,
            })
            continue

        hobby_rows.append({
            "email": email,
            "hobby_name": match.group(1),
            "priority": int(match.group(2)),
            "source": "excel",
        })

user_hobbies = pd.DataFrame(hobby_rows)

print("Hobby-Zuordnungen:", len(user_hobbies))
print("Ungültige Hobby-Werte:", len(invalid_hobbies))
print("Unterschiedliche Hobbys:", user_hobbies["hobby_name"].nunique())
print("Kleinste Priorität:", user_hobbies["priority"].min())
print("Größte Priorität:", user_hobbies["priority"].max())

print(
    "Doppelte Zuordnungen:",
    user_hobbies.duplicated(
        subset=["email", "hobby_name", "source"]
    ).sum()
)

Hobby-Zuordnungen: 4828
Ungültige Hobby-Werte: 0
Unterschiedliche Hobbys: 220
Kleinste Priorität: 0
Größte Priorität: 100
Doppelte Zuordnungen: 0
